# 🧠 Notebook 02: Feature Engineering & Customer Segmentation
## Advanced RFM Transformation, Target Formulations, and K-Means Clustering

**Author:** AI/ML Engineer  
**Objective:** Engineer leak-free historical behavioral features, formulate horizon targets, perform correlation analysis, and segment customers using K-Means with PCA visualization.


In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data.loader import load_transactions
from src.data.preprocessing import add_history_features, build_training_frame
from src.models.segmentation import compute_rfm_features, perform_customer_segmentation

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (10, 5)


### 1. Ingest Data & Engineer History Features
Historical features are calculated exclusively using transactions prior to or on the current transaction date, guaranteeing **zero data leakage**.


In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Retail_Transactions_Dataset.csv"
transactions, _ = load_transactions(DATA_PATH, sample_frac=0.1, use_cache=True)

# Derive history features
data_with_history = add_history_features(transactions)
data_with_history[[
    "Customer_Name", "Timestamp", "Total_Cost",
    "Transactions_So_Far", "Average_Spend_So_Far",
    "Days_Since_Previous_Purchase", "Unique_Products_So_Far"
]].head(10)


### 2. Feature Correlation Heatmap
Inspecting collinearity and relationships among numerical features.


In [ ]:
num_cols = [
    "Total_Items", "Total_Cost", "Discount_Flag",
    "Transactions_So_Far", "Average_Spend_So_Far",
    "Average_Items_So_Far", "Discount_Rate_So_Far",
    "Days_Since_Previous_Purchase", "Unique_Products_So_Far"
]

corr_matrix = data_with_history[num_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="Blues", cbar=True)
plt.title("Correlation Heatmap of Engineered Features")
plt.show()


### 3. Horizon Target Formulation
Evaluating label distributions across different time horizons (90, 180, 365 days).


In [ ]:
horizons = [90, 180, 365]
rates = []

for h in horizons:
    df_h = build_training_frame(transactions, horizon_days=h)
    pos_rate = df_h["Will_Purchase_In_Horizon"].mean()
    rates.append({"Horizon (Days)": h, "Sample Size": len(df_h), "Positive Return Rate": f"{pos_rate:.1%}"})

pd.DataFrame(rates)


### 4. RFM (Recency, Frequency, Monetary) Customer Segmentation
Applying unsupervised K-Means clustering with optimal cluster interpretation.


In [ ]:
rfm_results = perform_customer_segmentation(transactions, n_clusters=4, random_state=42)

print(f"Clustering Silhouette Score: {rfm_results['silhouette_score']:.3f}")
summary_df = rfm_results["segment_summary"]
summary_df


### 5. 2D PCA Visualization of Customer Segments


In [ ]:
rfm_df = rfm_results["rfm_df"]

plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=rfm_df.head(2000),
    x="PCA1",
    y="PCA2",
    hue="Segment",
    palette="tab10",
    alpha=0.7,
    s=35
)
plt.title("Customer Segments in 2D Latent Feature Space (PCA)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()
